In [15]:
import os

saved_model_path = "./yolov1_tensorflow"

if os.path.isfile(saved_model_path):
    os.remove(saved_model_path)

os.makedirs(saved_model_path, exist_ok=True)


In [16]:
import onnx2tf

onnx_path = "yolov1.onnx"

onnx2tf.convert(
    input_onnx_file_path=onnx_path,
    output_folder_path=saved_model_path,
    non_verbose=True
)

In [17]:
import tensorflow as tf

saved_model_path = "./yolov1_tensorflow"
tflite_ready_path = "./yolov1_saved_model_tflite"

model = tf.saved_model.load(saved_model_path)

@tf.function(input_signature=[tf.TensorSpec([1, 448, 448, 3], tf.float32, name="input")])
def serve_fn(input):
    return model(input)

tf.saved_model.save(model, tflite_ready_path, signatures={"serving_default": serve_fn})


In [18]:
!git clone https://github.com/byeongyun99/VOCdevkit.git

fatal: destination path 'VOCdevkit' already exists and is not an empty directory.


In [19]:
import tensorflow as tf
import numpy as np
from PIL import Image
import os

img_dir = "./VOCdevkit/VOC2007/JPEGImages"
img_files = [os.path.join(img_dir, f) for f in os.listdir(img_dir) if f.endswith(".jpg")]

def ResizePreprocess(img, size=(448, 448)):

    img = img.resize(size)
    img = np.array(img).astype(np.float32) / 255.0 
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    img = (img - mean) / std 
    
    img = np.expand_dims(img, axis=0)  # [1, 448, 448, 3]
    return img

def representative_dataset():
    for img_path in img_files[:100]:  
        img = Image.open(img_path).convert("RGB")
        img = ResizePreprocess(img)
        yield [img]

converter = tf.lite.TFLiteConverter.from_saved_model("./yolov1_saved_model_tflite")
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS]
converter.inference_input_type = tf.int8 
converter.inference_output_type = tf.int8
tflite_model = converter.convert()

tflite_model_file = "yolov1_int8.tflite"
with open(tflite_model_file, "wb") as f:
    f.write(tflite_model)

print(f"TFLite int8 model save {tflite_model_file}")    

TFLite int8 model save yolov1_int8.tflite
